In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!curl https://sdk.cloud.google.com | bash


In [ ]:
# !pip install datasets

In [ ]:
# from datasets import load_dataset
# dataset = load_dataset("pg19")

In [ ]:
# The Jungle book by Rudyard kipling
import requests
content = requests.get("https://www.gutenberg.org/files/236/236-0.txt").text
with open("The_Jungle_book.txt", "w", encoding="utf-8") as f:
  f.write(content)

In [ ]:
print(content[:1000])

ï»¿The Project Gutenberg EBook of The Jungle Book, by Rudyard Kipling

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org


Title: The Jungle Book

Author: Rudyard Kipling

Release Date: January 16, 2006 [EBook #236]
Last Updated: October 6, 2016

Language: English

Character set encoding: UTF-8

*** START OF THIS PROJECT GUTENBERG EBOOK THE JUNGLE BOOK ***




Produced by An Anonymous Volunteer and David Widger





THE JUNGLE BOOK

By Rudyard Kipling



Contents

     Mowgliâs Brothers
     Hunting-Song of the Seeonee Pack
     Kaaâs Hunting
     Road-Song of the Bandar-Log
     âTiger! Tiger!â
      Mowgliâs Song
     The White Seal
     Lukannon
     âRikki-Tikki-Taviâ
      Darzeeâs Chant
     Toomai of the Elephants
     


In [ ]:
from string import punctuation

import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm_notebook


import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torch.nn import functional as F

In [ ]:
sequence_length = 100
BATCH_SIZE = 128
EPOCHS = 30

# dataset file path
FILE_PATH = "/content/The_Jungle_book.txt"

# read the data
text = open(FILE_PATH, encoding="utf-8").read()

# remove caps, comment this code if you want uppercase characters as well
text = text.lower()
# remove punctuation
# text = text.translate(str.maketrans("", "", 'punctuation'))

In [ ]:
text[:300]

'ï»¿the project gutenberg ebook of the jungle book, by rudyard kipling\n\nthis ebook is for the use of anyone anywhere at no cost and with\nalmost no restrictions whatsoever.  you may copy it, give it away or\nre-use it under the terms of the project gutenberg license included\nwith this ebook or online a'

In [ ]:
# print some stats
n_chars = len(text)
vocab = ''.join(sorted(set(text)))
print("unique_chars:", vocab)
n_unique_chars = len(vocab)
print("Number of characters:", n_chars)
print("Number of unique characters:", n_unique_chars)

unique_chars: 
 !#$%()*,-./0123456789:;?@[]`abcdefghijklmnopqrstuvwxyz»¿âï
Number of characters: 298571
Number of unique characters: 65


In [ ]:
vocab

'\n !#$%()*,-./0123456789:;?@[]`abcdefghijklmnopqrstuvwxyz\x80\x98\x99\x9c\x9d»¿âï'

In [ ]:
# dictionary that converts characters to integers
char2int = {c: i for i, c in enumerate(vocab)}
# dictionary that converts integers to characters
int2char = {i: c for i, c in enumerate(vocab)}

In [ ]:
print(char2int)

{'\n': 0, ' ': 1, '!': 2, '#': 3, '$': 4, '%': 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '/': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, ';': 24, '?': 25, '@': 26, '[': 27, ']': 28, '`': 29, 'a': 30, 'b': 31, 'c': 32, 'd': 33, 'e': 34, 'f': 35, 'g': 36, 'h': 37, 'i': 38, 'j': 39, 'k': 40, 'l': 41, 'm': 42, 'n': 43, 'o': 44, 'p': 45, 'q': 46, 'r': 47, 's': 48, 't': 49, 'u': 50, 'v': 51, 'w': 52, 'x': 53, 'y': 54, 'z': 55, '\x80': 56, '\x98': 57, '\x99': 58, '\x9c': 59, '\x9d': 60, '»': 61, '¿': 62, 'â': 63, 'ï': 64}


In [ ]:
print(int2char)

{0: '\n', 1: ' ', 2: '!', 3: '#', 4: '$', 5: '%', 6: '(', 7: ')', 8: '*', 9: ',', 10: '-', 11: '.', 12: '/', 13: '0', 14: '1', 15: '2', 16: '3', 17: '4', 18: '5', 19: '6', 20: '7', 21: '8', 22: '9', 23: ':', 24: ';', 25: '?', 26: '@', 27: '[', 28: ']', 29: '`', 30: 'a', 31: 'b', 32: 'c', 33: 'd', 34: 'e', 35: 'f', 36: 'g', 37: 'h', 38: 'i', 39: 'j', 40: 'k', 41: 'l', 42: 'm', 43: 'n', 44: 'o', 45: 'p', 46: 'q', 47: 'r', 48: 's', 49: 't', 50: 'u', 51: 'v', 52: 'w', 53: 'x', 54: 'y', 55: 'z', 56: '\x80', 57: '\x98', 58: '\x99', 59: '\x9c', 60: '\x9d', 61: '»', 62: '¿', 63: 'â', 64: 'ï'}


In [ ]:
text[5000:6000]

' singsong whine of a tiger who has\ncaught nothing and does not care if all the jungle knows it.\n\nâ\x80\x9cthe fool!â\x80\x9d said father wolf. â\x80\x9cto begin a nightâ\x80\x99s work with that noise!\ndoes he think that our buck are like his fat waingunga bullocks?â\x80\x9d\n\nâ\x80\x9châ\x80\x99sh. it is neither bullock nor buck he hunts to-night,â\x80\x9d said mother\nwolf. â\x80\x9cit is man.â\x80\x9d\n\nthe whine had changed to a sort of humming purr that seemed to come\nfrom every quarter of the compass. it was the noise that bewilders\nwoodcutters and gypsies sleeping in the open, and makes them run\nsometimes into the very mouth of the tiger.\n\nâ\x80\x9cman!â\x80\x9d said father wolf, showing all his white teeth. â\x80\x9cfaugh! are there\nnot enough beetles and frogs in the tanks that he must eat man, and on\nour ground too!â\x80\x9d\n\nthe law of the jungle, which never orders anything without a reason,\nforbids every beast to eat man except when he is killing to show hi

In [ ]:
# convert all text into integers
encoded_text = np.array([char2int[c] for c in text])

In [ ]:
# convert all text to integers
encoded_text =np.array([char2int[c] for c in text])

In [ ]:
len(text)

298571

In [ ]:
len(encoded_text)

298571

In [ ]:
text[:25]

'ï»¿the project gutenberg '

In [ ]:
encoded_text[:25]

array([64, 61, 62, 49, 37, 34,  1, 45, 47, 44, 39, 34, 32, 49,  1, 36, 50,
       49, 34, 43, 31, 34, 47, 36,  1])

In [ ]:
# tokenize  input_seq, target_seq
def create_sequence_data(text, sequence_length):
  input_seq = []
  target_seq = []
  for idx in range(0, len(text), sequence_length):
    st_idx = idx
    end_idx = st_idx + sequence_length + 1
    if end_idx > len(text):
      # Exclude last slice that may not be full
      continue
    input_seq.append(text[st_idx:end_idx-1])
    target_seq.append(text[st_idx+1:end_idx])
  return input_seq, target_seq

In [ ]:
input_seq, target_seq = create_sequence_data(encoded_text, sequence_length=sequence_length)

In [ ]:
input_seq[0:3]

[array([64, 61, 62, 49, 37, 34,  1, 45, 47, 44, 39, 34, 32, 49,  1, 36, 50,
        49, 34, 43, 31, 34, 47, 36,  1, 34, 31, 44, 44, 40,  1, 44, 35,  1,
        49, 37, 34,  1, 39, 50, 43, 36, 41, 34,  1, 31, 44, 44, 40,  9,  1,
        31, 54,  1, 47, 50, 33, 54, 30, 47, 33,  1, 40, 38, 45, 41, 38, 43,
        36,  0,  0, 49, 37, 38, 48,  1, 34, 31, 44, 44, 40,  1, 38, 48,  1,
        35, 44, 47,  1, 49, 37, 34,  1, 50, 48, 34,  1, 44, 35,  1]),
 array([30, 43, 54, 44, 43, 34,  1, 30, 43, 54, 52, 37, 34, 47, 34,  1, 30,
        49,  1, 43, 44,  1, 32, 44, 48, 49,  1, 30, 43, 33,  1, 52, 38, 49,
        37,  0, 30, 41, 42, 44, 48, 49,  1, 43, 44,  1, 47, 34, 48, 49, 47,
        38, 32, 49, 38, 44, 43, 48,  1, 52, 37, 30, 49, 48, 44, 34, 51, 34,
        47, 11,  1,  1, 54, 44, 50,  1, 42, 30, 54,  1, 32, 44, 45, 54,  1,
        38, 49,  9,  1, 36, 38, 51, 34,  1, 38, 49,  1, 30, 52, 30]),
 array([54,  1, 44, 47,  0, 47, 34, 10, 50, 48, 34,  1, 38, 49,  1, 50, 43,
        33, 34, 47,  1, 

In [ ]:
target_seq[0:3]

In [ ]:
train_seq, valid_seq, train_targets, valid_targets = train_test_split(input_seq, target_seq, test_size=0.1)

In [ ]:
len(input_seq), len(target_seq)

(2985, 2985)

In [ ]:
def one_hot_encode(sequence, vocab_size):
    # Creating a multi-dimensional array of zeros with the desired output shape
    # (Sequence Length, One-Hot Encoding Size)
    seq_length = len(sequence)
    output = np.zeros((seq_length, vocab_size), dtype=np.float32)

    for seq in range(seq_length):
      output[seq, sequence[seq]] = 1

    return output


In [ ]:
one_hot_encode([1, 2, 38], len(vocab)).shape

(3, 65)

In [ ]:
class TextGenDataset(Dataset):
  def __init__(self, text_seq, target_seq, seq_length, vocab_size):
    super().__init__()
    self.text_seq = text_seq
    self.target_seq = target_seq
    self.seq_length = seq_length
    self.vocab_size = vocab_size

  def __getitem__(self, idx):
    return one_hot_encode(self.text_seq[idx], self.vocab_size), self.target_seq[idx]

  def __len__(self):
    return len(self.text_seq)

train_ds = TextGenDataset(train_seq, train_targets, sequence_length, len(vocab))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=False, drop_last=True)

valid_ds = TextGenDataset(valid_seq, valid_targets, sequence_length, len(vocab))
valid_dl = DataLoader(valid_ds, batch_size=64, drop_last=True)

In [ ]:
x, y = train_ds[0]
x.shape, y.shape

((100, 65), (100,))

In [ ]:
print(x)
print(y)

In [ ]:
for x, y in train_dl:
  break

In [ ]:
x.shape, y.shape

(torch.Size([128, 100, 65]), torch.Size([128, 100]))

In [ ]:
print(x[0][2])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [ ]:
class Model(nn.Module):
    def __init__(self, input_size, output_size, hidden_dim, n_layers):
        super(Model, self).__init__()

        # Defining some parameters
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        #Defining the layers
        # RNN Layer
        self.rnn = nn.GRU(input_size, hidden_dim, n_layers, batch_first=True)
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x, hidden=None):

        batch_size = x.size(0)

        # #Initializing hidden state for first input using method defined below
        # hidden = self.init_hidden(batch_size)

        # Passing in the input and hidden state into the model and obtaining outputs
        out, hidden = self.rnn(x, hidden)

        # Reshaping the outputs such that it can be fit into the fully connected layer
        out = out.contiguous().view(-1, self.hidden_dim)
        out = self.fc(out)
        return out, hidden.detach()


In [ ]:
# Instantiate the model with hyperparameters
model = Model(input_size=len(vocab), output_size=len(vocab), hidden_dim=512, n_layers=3)

# Define hyperparameters
n_epochs = 100
lr=0.001
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define Loss, Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
# Training Run
def train(model, train_dl, criterion, optimizer, batch_size, device=None, n_epochs=100):
  if device is None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
  model = model.to(device)
  model.train()
  criterion = criterion.to(device)
  for epoch in range(1, n_epochs + 1):
    hidden = None
    n_batches = 0
    losses = 0
    for input_seq, target_seq in train_dl:
      input_seq, target_seq = input_seq.to(device), target_seq.to(device)
      optimizer.zero_grad() # Clears existing gradients from previous epoch
      output, hidden = model(input_seq, hidden)
      loss = criterion(output, target_seq.view(-1).long())
      loss.backward() # Does backpropagation and calculates gradients
      optimizer.step() # Updates the weights accordingly
      # hidden.detach_()
      losses += loss.item()
      n_batches += 1

    if epoch%10 == 0:
      print('Epoch: {}/{}.............'.format(epoch, n_epochs), end=' ')
      print("Loss: {:.4f}".format(losses / n_batches))
  return model

In [ ]:
model = train(model, train_dl, criterion, optimizer,BATCH_SIZE, device, n_epochs=200)

Epoch: 10/200............. Loss: 1.8891
Epoch: 20/200............. Loss: 1.3710
Epoch: 30/200............. Loss: 1.1425
Epoch: 40/200............. Loss: 0.9744
Epoch: 50/200............. Loss: 0.8479
Epoch: 60/200............. Loss: 0.6812
Epoch: 70/200............. Loss: 0.5519
Epoch: 80/200............. Loss: 0.4730
Epoch: 90/200............. Loss: 0.3432
Epoch: 100/200............. Loss: 0.2695
Epoch: 110/200............. Loss: 0.2445
Epoch: 120/200............. Loss: 0.1034
Epoch: 130/200............. Loss: 0.0926
Epoch: 140/200............. Loss: 0.0470
Epoch: 150/200............. Loss: 0.0202
Epoch: 160/200............. Loss: 0.0083
Epoch: 170/200............. Loss: 0.0066
Epoch: 180/200............. Loss: 0.0055
Epoch: 190/200............. Loss: 0.0047
Epoch: 200/200............. Loss: 0.0041


In [ ]:
#Save Model Weights (Download it for future use)
state = {
    "model": model.state_dict(),
    "char2int": char2int,
    "int2char": int2char,
    "n_layers": 3,
    "hidden_dim": 512
}
torch.save(state, "./model_jungle_book.pt")


In [ ]:
# Load the saved model
state = torch.load("./model_jungle_book.pt")
model.load_state_dict(state["model"])

<All keys matched successfully>

In [ ]:
# Load the saved model
state = torch.load("./model_jungle_book.pt")
model.load_state_dict(state["model"])

<All keys matched successfully>

In [ ]:
def predict(model, hidden, character, char2int, int2char, device):
    # One-hot encoding our input to fit into the model
    # print(character)
    character = np.array([char2int[c] for c in character])
    # print(character)
    character = one_hot_encode(character, vocab_size=len(char2int))
    # print(character.shape)
    character = torch.from_numpy(character).unsqueeze(0).to(device)
    with torch.no_grad():
      out, hidden = model(character)
    # print(hidden.size())
    prob = nn.functional.softmax(out[-1], dim=0).data
    # Taking the class with the highest probability score from the output
    char_ind = torch.max(prob, dim=0)[1].item()

    return int2char[char_ind], hidden

In [ ]:
def sample(model, char2int, int2char, out_len, start='hey', device=None):
    model.eval() # eval mode
    if device is None:
      device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    start = start.lower()
    # First off, run through the starting characters
    chars = [ch for ch in start]
    size = out_len - len(chars)
    # Now pass in the previous characters and get a new one
    hidden = None
    for ii in range(size):
        # print(chars)
        char, hidden = predict(model, hidden, chars, char2int, int2char, device)
        chars.append(char)

    return ''.join(chars)

In [ ]:
print(sample(model, char2int, int2char, out_len=1500, start="Run Mowgli"))

run mowgli the frog i will call thee--the time will
come when thou hast seen the elephants dance, and then i will let thee go into all the keddahs.â

there we dreamed bad dreams in the night,
and we were very much afraid. i am only a baggage camel of the 39th native
infantry, instead of running all round the camp?â said the mule, âor youâll snap your long stick-legs betwee had hit him in
the mouth. ârun back, messua. this is one of the foolish tales they tell
under them. so he evening i have lain here listening,â he
called back over his shoulder, âand, except once or twice that shere khan was not a creature
to be trusted, and that some day he must never touch cattle because he had
been bought in the same things. ânow, you gentlemen were alarmed, i believe, when i trumpeted.â

ânot alark so that they may attend to hear any more about a foot and bring together again with
a whole bushel of seaweed between the splits. they mowgli repeated, with the kiteâs whistle at t